In [ ]:
# ============================================================
# CHẠY TIẾP MỘT SEED BỊ DỪNG Ở MỐC AN TOÀN (CHỈ DÙNG KHI CÓ CHUA_XONG.txt)
# Thêm Input: dataset h3-gop-3tap-tsvm VÀ dataset chứa last_resume.pt của seed đó (tạo từ zip kết quả phiên trước).
# ============================================================
MODEL = "tsvm"             # sửa theo seed bị dừng
SEEDS = [0]                # đúng 1 seed bị dừng
CHAY_THU = False
GIOI_HAN_GIO = 11.8
DU_PHONG_GIO = 0.25
assert len(SEEDS) == 1


In [ ]:
# ============================================================
# CELL 1: TÌM INPUT, NẠP ULTRALYTICS BẢN TSVM_FP32, TẠO DATA YAML (TRAIN / VAL / TEST)
# ============================================================
import os, sys, glob, time, json, shutil, yaml
T_PHIEN = time.time()

def tim(mau):
    return sorted(glob.glob(mau, recursive=True))

_repo = tim("/kaggle/input/**/ultralytics_TSVM_FP32/PATCH_FP32.md")
_data = tim("/kaggle/input/**/H3_Gop_3tap/images/test")
assert _repo, "❌ Chưa thêm dataset chứa thư mục ultralytics_TSVM_FP32"
assert _data, "❌ Chưa thêm dataset h3-gop-3tap-tsvm (thư mục H3_Gop_3tap)"
CUSTOM_REPO_PATH = os.path.dirname(_repo[0])
DATASET_ROOT = os.path.dirname(os.path.dirname(_data[0]))

if not os.path.exists("/kaggle/working/ultralytics"):
    os.symlink(CUSTOM_REPO_PATH, "/kaggle/working/ultralytics")
for p in ["/kaggle/working", CUSTOM_REPO_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

import ultralytics
assert ultralytics.__version__ == "8.4.127", ultralytics.__version__
assert ultralytics.__file__.startswith("/kaggle/working/ultralytics"), ultralytics.__file__
from ultralytics.nn.modules import C2TSVMamba
print(f"✅ Ultralytics {ultralytics.__version__} (bản TSVM_FP32) từ {ultralytics.__file__}")

dem = {s: len(os.listdir(f"{DATASET_ROOT}/images/{s}")) for s in ("train", "val", "test")}
dem_nhan = {s: len(os.listdir(f"{DATASET_ROOT}/labels/{s}")) for s in ("train", "val", "test")}
assert dem == {"train": 2455, "val": 532, "test": 522} and dem_nhan == {"train": 2455, "val": 532, "test": 522}, (dem, dem_nhan)
print("📁 Dữ liệu:", DATASET_ROOT, dem)

DATA_YAML = "/kaggle/working/data_h3_3tap.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": DATASET_ROOT, "train": "images/train", "val": "images/val", "test": "images/test",
                    "nc": 1, "names": {0: "polyp"}}, f, sort_keys=False, allow_unicode=True)

if MODEL == "tsvm":
    SRC_YAML = f"{CUSTOM_REPO_PATH}/cfg/models/26/yolo26-seg-TopologyShapeVMamba.yaml"
    # Tên tệp PHẢI bắt đầu bằng "yolo26s": Ultralytics đoán scale từ tên tệp
    MODEL_YAML = "/kaggle/working/yolo26s-seg-TopologyShapeVMamba.yaml"
    with open(SRC_YAML, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["scale"] = "s"
    with open(MODEL_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    TEN = "TSVM_FP32"
else:
    MODEL_YAML = None
    TEN = "YOLO26s_seg"


In [ ]:
# ============================================================
# CELL 2: TẢI TRỌNG SỐ yolo26s-seg.pt (CẦN Settings > Internet: ON)
# ============================================================
WEIGHT_PATH = "/kaggle/working/yolo26s-seg.pt"
URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt"
if os.path.exists(WEIGHT_PATH) and os.path.getsize(WEIGHT_PATH) < 20 * 1024 * 1024:
    os.remove(WEIGHT_PATH)
if not os.path.exists(WEIGHT_PATH):
    !curl -L --retry 5 --retry-delay 2 -C - "{URL}" -o "{WEIGHT_PATH}"
size_mb = os.path.getsize(WEIGHT_PATH) / 2**20
print(f"Dung lượng: {size_mb:.2f} MB")
assert size_mb > 20.0, "❌ Tải chưa đủ, chạy lại cell này (Settings > Internet: ON)"


In [ ]:
# ============================================================
# CELL 3: KIỂM TRA CẤU TRÚC MÔ HÌNH TRƯỚC KHI TRAIN (~10 GIÂY)
# ============================================================
from ultralytics import YOLO
_m = YOLO(MODEL_YAML).load(WEIGHT_PATH) if MODEL == "tsvm" else YOLO(WEIGHT_PATH)
l10 = _m.model.model[10]
n_all = sum(p.numel() for p in _m.model.parameters())
n_l10 = sum(p.numel() for p in l10.parameters())
print(f"Tầng 10: {type(l10).__name__}, {n_l10:,} tham số; toàn mô hình {n_all:,}")
if MODEL == "tsvm":
    assert type(l10).__name__ == "C2TSVMamba" and n_l10 == 1_811_201, "❌ Tầng 10 không đúng TSVM đầy đủ"
else:
    assert type(l10).__name__ == "C2PSA", "❌ Baseline phải giữ C2PSA ở tầng 10"
print("✅ Đúng cấu trúc")
del _m


In [ ]:
# ============================================================
# CELL 4 (CHẠY TIẾP): resume=True từ last_resume.pt, giữ nguyên tên lượt, rồi đánh giá test
# ============================================================
import torch
SEED = SEEDS[0]
RUN_NAME = f"H3_Gop_3tap_{TEN}_s{SEED}"
_ck = [p for p in tim("/kaggle/input/**/last_resume.pt") if f"_s{SEED}" in p or len(tim("/kaggle/input/**/last_resume.pt")) == 1]
assert len(_ck) == 1, f"❌ Cần đúng 1 last_resume.pt của seed {SEED}: {_ck}"
run_dir = f"/kaggle/working/runs/{RUN_NAME}"
os.makedirs(f"{run_dir}/weights", exist_ok=True)
# chép cả thư mục lượt cũ (results.csv, best.pt, args.yaml...) để Ultralytics ghi tiếp vào đúng chỗ
_src = os.path.dirname(os.path.dirname(_ck[0]))
shutil.copytree(_src, run_dir, dirs_exist_ok=True)
shutil.copy2(_ck[0], f"{run_dir}/weights/last.pt")
ck = torch.load(f"{run_dir}/weights/last.pt", map_location="cpu", weights_only=False)
ck["train_args"].update(data=DATA_YAML, project="/kaggle/working/runs", name=RUN_NAME, save_dir=run_dir)
torch.save(ck, f"{run_dir}/weights/last.pt")
print("Chạy tiếp từ epoch", ck["epoch"] + 2, "/ 100")
t0 = time.time()
YOLO(f"{run_dir}/weights/last.pt").train(resume=True)
best = YOLO(f"{run_dir}/weights/best.pt")
m = best.val(data=DATA_YAML, split="test", imgsz=640, batch=8, half=False, device=0, plots=False,
             project="/kaggle/working/runs", name=RUN_NAME + "_test", exist_ok=True)
kq = {k: float(v) for k, v in m.results_dict.items()}
kq.update(seed=SEED, model=TEN, gio_phien_chay_tiep=(time.time() - t0) / 3600, chay_tiep=True,
          gpu=torch.cuda.get_device_name(0), torch=torch.__version__)
json.dump(kq, open(f"{run_dir}/test_metrics.json", "w"), indent=1)
if os.path.exists(f"{run_dir}/CHUA_XONG.txt"):
    os.rename(f"{run_dir}/CHUA_XONG.txt", f"{run_dir}/DA_CHAY_TIEP.txt")
shutil.make_archive(f"/kaggle/working/runs_{RUN_NAME}", "zip", "/kaggle/working/runs", RUN_NAME)
print(f"✅ seed {SEED} chạy tiếp xong: test mask mAP50-95 {kq.get('metrics/mAP50-95(M)', float('nan')):.4f}")
